# Prova de Conceito (PoC): Normalização Semântica de Dados Fiscais via LLM

**Missão de negócio.** Reduzir ambiguidade no campo `XPROD` e produzir atributos fiscais confiáveis (`Nome_Normalizado`, `Quantidade`, `Unidade_Medida`) com rastreabilidade técnica.

**Hipótese central.** Uma arquitetura em blocos, com recuperação lexical eficiente (BM25), extração estruturada por LLM e validação determinística, supera abordagens ad-hoc e reduz riscos de baixa qualidade operacional.

**O que este notebook entrega.**

1. desenho metodológico com justificativa computacional;
2. implementação ponta a ponta do pipeline (Blocagem → LLM → Validação → Benchmark);
3. avaliação científica com log de erros para melhoria contínua;
4. fluxo reprodutível para auditoria técnica e evolução de produto.

---

> **Critério desta PoC:** toda decisão deve ser explicável, mensurável e replicável.

## 1. O Desafio Computacional: A Maldição de $\mathcal{O}(n^2)$

O objetivo final é agrupar descrições que representam o mesmo produto físico. A abordagem ingênua seria comparar cada nota fiscal com todas as demais, usando métricas de distância textual (como Levenshtein ou Jaccard).

Se temos $N$ notas fiscais, o total de comparações aos pares é:

$$
C_{N,2} = \frac{N(N-1)}{2}
$$

Isso leva a uma complexidade temporal $\mathcal{O}(n^2)$. Em escala estadual, com 10 milhões de notas, o número de pares chega à ordem de **dezenas de trilhões**, tornando a estratégia de força bruta computacionalmente e financeiramente inviável.

## 2. Bloco 1: Blocagem Semântica com Algoritmo BM25

Para contornar a complexidade quadrática, o *Documento de Arquitetura de Solução* (ADR-004) propõe o uso do algoritmo **BM25 (Best Matching 25)**.

O BM25 atua como um motor de redução do espaço de busca. Após a indexação, a busca se torna sublinear na prática para recuperação de candidatos. Em vez de usar embeddings densos (que aumentariam custo computacional e operacional nesta fase), o BM25 aplica um modelo probabilístico eficiente baseado em Bag-of-Words avançado.

A função de pontuação que ranqueia a similaridade de um documento $D$ para uma query $Q$ (com termos $q_i$) é:

$$
\text{Score}(D, Q) = \sum_{i=1}^{n} \text{IDF}(q_i) \cdot \frac{f(q_i, D) \cdot (k_1 + 1)}{f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}
$$

Onde:
- **$\text{IDF}(q_i)$** penaliza termos muito comuns (ex.: “KG”, “LT”) e valoriza termos discriminativos (ex.: “BOZZANO”).
- **$f(q_i, D)$** é a frequência do termo no documento.
- **$b$** e **$k_1$** são hiperparâmetros de normalização e saturação.

Neste PoC, o Bloco 1 lê descrições brutas, aplica o BM25 e gera lotes semânticos prontos para o LLM.

## 3. Preparando o Cérebro: Batching e Mecanismo de Atenção

Por que não enviar as notas uma a uma para a IA? O documento arquitetural (ADR-001) define que a saída do Bloco 1 deve ser um lote (*batch*) estrito de **15 itens similares**.

Isso resolve dois problemas simultaneamente:

1. **Evita o *Lazy Evaluation***: estudos indicam que LLMs podem truncar respostas e aumentar alucinações quando os lotes são extensos. O limite de 15 itens tende a equilibrar custo de tokens e qualidade semântica.
2. **Explora o *Self-Attention* da rede neural**: descrições fiscais curtas (*short-texts*) costumam ter pouco contexto sintático. Quando o modelo recebe um lote temático (agrupado pelo BM25), ele ganha âncoras relacionais mais fortes entre os termos.

Exemplo prático: se a string `ESP BARBA PRESTOBARB` for enviada isolada, a sigla `ESP` é ambígua. Porém, dentro de um lote com ocorrências como `ESPUMA BARBA BOZZANO`, o modelo consegue inferir, pelo contexto cruzado, que `ESP` provavelmente significa **Espuma**.

Abaixo, implementamos a camada de domínio e o caso de uso (Clean Architecture) para materializar esse motor.

In [17]:
import pandas as pd
import re
from rank_bm25 import BM25Okapi
from typing import List

In [18]:
# ---------------------------------------------------------
# CAMADA 1: DOMÍNIO E PRÉ-PROCESSAMENTO (Clean Text)
# ---------------------------------------------------------
class TextNormalizer:
    """Responsável por limpar e padronizar strings do campo XPROD."""

    @staticmethod
    def tokenize(text: str) -> List[str]:
        if not isinstance(text, str):
            return []
        text = text.lower().strip()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        return text.split()


# ---------------------------------------------------------
# CAMADA 2: CASO DE USO (Blocagem com BM25)
# ---------------------------------------------------------
class SemanticBlocker:
    """Motor de indexação BM25 para recuperar lotes semânticos."""

    def __init__(self, data: pd.DataFrame, text_column: str):
        if text_column not in data.columns:
            raise ValueError(f"Coluna '{text_column}' não encontrada no DataFrame.")

        self.data = data.copy()
        self.text_column = text_column
        self.corpus_tokens: List[List[str]] = []
        self.bm25_engine = None

    def fit(self):
        print("Iniciando ingestão e tokenização...")

        textos = self.data[self.text_column].fillna('').astype(str).tolist()
        self.corpus_tokens = [TextNormalizer.tokenize(doc) for doc in textos]

        print("Treinando índice invertido BM25...")
        self.bm25_engine = BM25Okapi(self.corpus_tokens)
        print("Motor de blocagem pronto.")

    def get_batch_for(self, query_text: str, batch_size: int = 15) -> List[str]:
        """Retorna um lote com os itens mais similares para a consulta."""
        if self.bm25_engine is None:
            raise ValueError("Execute .fit() antes de usar get_batch_for().")

        if batch_size <= 0:
            raise ValueError("batch_size deve ser maior que zero.")

        tokenized_query = TextNormalizer.tokenize(query_text)

        top_docs = self.bm25_engine.get_top_n(
            query=tokenized_query,
            documents=self.data[self.text_column].fillna('').astype(str).tolist(),
            n=batch_size
        )
        return top_docs

In [19]:
# 1) Ingestão de dados
DATA_PATH = '../data/AmostraDescMerc.csv'
df_sefaz = pd.read_csv(DATA_PATH, sep='\t')

print(f"Base carregada: {len(df_sefaz):,} linhas | colunas: {list(df_sefaz.columns)}")

# 2) Treino do Bloco 1 (BM25)
blocker = SemanticBlocker(data=df_sefaz, text_column='Text')
blocker.fit()

# 3) Teste rápido de recuperação
descricao_alvo = 'BOZZANO ESPUMA BARBA CERRADA 6X190G'
lote_para_o_llm = blocker.get_batch_for(query_text=descricao_alvo, batch_size=15)

print("\nLote montado para envio ao LLM:")
display(pd.DataFrame({'rank': range(1, len(lote_para_o_llm) + 1), 'XPROD_recuperado': lote_para_o_llm}))

Base carregada: 15,000 linhas | colunas: ['Text', 'Category']
Iniciando ingestão e tokenização...
Treinando índice invertido BM25...
Motor de blocagem pronto.

Lote montado para envio ao LLM:


,rank,XPROD_recuperado
0,1,BOZZANO ESPUMA BARBA CERRADA 6X190G
1,2,PINCEL BARBA MUNDIAL
2,3,GRECIN 5 CASTANHO ESC.BARBA
3,4,DESODORANTE BOZZANO AEROSOL FRESH 150ML
4,5,ESP BARBA PRESTOBARB P-NORM 150GR
5,6,POS BARBA BARBASOL BALSAMO PACIFIC 100ML
6,7,LOCAO POS BARBA - 120ML - Lote: 4857
7,8,CR BARBA GIL ALOE HIDRATANTE 65G
8,9,DES.BOZZANO AERO A.SPORT 90G
9,10,LOCAO AGUA NOVA NIGHTBLUE 100ML BOZZANO


## 3.5 Validação de Domínio da Amostra (Análise Exploratória)

Antes de avaliar a qualidade da blocagem, precisamos verificar se a amostra contém o domínio esperado para os testes.

Em recuperação de informação, há um efeito clássico de *out-of-vocabulary* (OOV): quando o termo da consulta não está bem representado no corpus, o BM25 tende a recuperar itens por sinais secundários (por exemplo, volume, unidade ou padrão de embalagem), e não pelo núcleo semântico do produto.

Por isso, nesta etapa vamos:
- inspecionar a distribuição de categorias da base;
- identificar um domínio com maior variação textual (ruído real);
- selecionar consultas representativas para o experimento controlado do Bloco 1.

In [20]:
print("Distribuição de categorias na amostra:")
display(df_sefaz['Category'].value_counts().to_frame('qtd_registros'))

print("\nBuscando candidatos ruidosos para o Teste C (domínio cosméticos)...")

filtro_shampoo = df_sefaz[
    df_sefaz['Text'].str.contains(r'\bSHAMPOO\b|\bSH\b', na=False, case=False, regex=True)
]

n_amostra = min(10, len(filtro_shampoo))
print(f"Total de linhas candidatas: {len(filtro_shampoo)} | exibindo {n_amostra} exemplos")

amostra = filtro_shampoo['Text'].dropna().sample(n=n_amostra, random_state=42)
for i, texto in enumerate(amostra.values, 1):
    print(f"[{i:02d}] {texto}")

Distribuição de categorias na amostra:


,qtd_registros
Category,
cosmeticos,5000
matlimpeza,5000
alimentos,5000



Buscando candidatos ruidosos para o Teste C (domínio cosméticos)...
Total de linhas candidatas: 322 | exibindo 10 exemplos
[01] SH ELSEVE OLEO EXTRAORD NUTRICAO 1X400ML RH1325600
[02] ABACATE POS QUIMICA SH 250ML
[03] PACK SHAMPOO COND MARCIA CORES LUMINOSAS 6X1
[04] SHAMPOO ALOE FRUTAS LOTE 15
[05] SH PANTENE COR RADIANTE 400ML
[06] SHAMPOO SEDA PRETOS LUMINOSOS 350ML
[07] PH SMITH SH CURL LOCKS
[08] SHAMPOO FRUCTIS FRESCOR VITAMINADO 400ML
[09] TUTANNOL FORTALECIMENTO SHAMPOO 300ML CX C/12
[10] SPLASH SH 1 20 4X5 LTS - ESPECIAL


## 4. Validação do Bloco 1 com Casos Reais

Agora vamos testar a blocagem semântica com três cenários que capturam ruídos típicos de descrições fiscais:

1. **Abreviação e truncamento**: nome do produto incompleto ou reduzido.
2. **Ruído estrutural**: presença de metadados e padrões não semânticos na string.
3. **Variação lexical no mesmo item**: múltiplas formas de escrita para o mesmo produto comercial.

O objetivo desta etapa é verificar se o BM25 consegue retornar, para cada consulta, um lote de 15 descrições semanticamente próximas. Esse lote será a entrada do LLM nas próximas camadas, onde ocorrerá a normalização final.

Em termos operacionais, buscamos maximizar *recall* de candidatos no Bloco 1 e deixar a decisão fina de canonização para as etapas seguintes.

### 4.1 Protocolo de Leitura dos Resultados

Para cada consulta, avalie o lote retornado com três critérios simples:

- **Coerência semântica**: quantos itens do lote descrevem o mesmo produto ou família próxima.
- **Robustez a ruído**: presença de variantes com abreviações, erros de digitação e padrões de sistema.
- **Utilidade para o LLM**: se o conjunto oferece contexto suficiente para a etapa de canonização textual.

Se esses três critérios forem atendidos, o Bloco 1 está cumprindo seu papel: reduzir o espaço de busca e entregar contexto estruturado para as próximas camadas.

In [21]:
exemplos_de_ouro = {
    'A. Teste de Abreviação (Amaciantes)': 'AMACIANTE BABY SOFT 2 LITROS',
    'B. Teste de Ruído Extremo (Iogurtes)': 'IOGURTE LIQUIDO MORANGO VIGOR',
    'C. Prova de Robustez (Cosméticos)': 'SHAMPOO ELSEVE OLEO EXTRAORDINARIO NUTRICAO 400ML'
}

for titulo, query in exemplos_de_ouro.items():
    print(f"\n{'=' * 80}")
    print(f"{titulo}")
    print(f"Consulta: {query}")
    print(f"{'=' * 80}")

    lote = blocker.get_batch_for(query_text=query, batch_size=15)
    display(pd.DataFrame({'rank': range(1, len(lote) + 1), 'XPROD_recuperado': lote}))


A. Teste de Abreviação (Amaciantes)
Consulta: AMACIANTE BABY SOFT 2 LITROS


,rank,XPROD_recuperado
0,1,EQUILIBRIO AMACIANTE 02X05 LITROS
1,2,EQUILIBRIO COTTON SOFT 06X02 LITROS
2,3,AMACIANTE 5 LITROS 4X5L MACIEZ
3,4,AMAC BABY SOFT 5LI AZUL
4,5,AMAC BABY SOFT T. CUIDADO BRANCO 2 LT CX/0006
5,6,AMAC BABY SOFT T. CARINHO AZUL 2 LT CX/0006
6,7,AMACIANTE NANO SOFT CARINHO 6X2L
7,8,AMACIANTE NANO SOFT AMOR 6X2L
8,9,AMAC BABY SOFT 5LI FLOWERS ROSA
9,10,AMACIANTE NANO SOFT ENCANTO 6X2L AZUL



B. Teste de Ruído Extremo (Iogurtes)
Consulta: IOGURTE LIQUIDO MORANGO VIGOR


,rank,XPROD_recuperado
0,1,DANONINHO CRUSH MORANGO IOGURTE 400GX6
1,2,TORTUGUITA IOGURTE MORANGO DI 408G
2,3,IOGURTE GREGO MORANGO BATAVO 100G
3,4,IOGURTE PAULISTA POLPA MORANGO 600GR
4,5,IOGURTE DANONE 900G DANONINHO MORANGO
5,6,36-IOGURTE INT MAMAO VIGOR 12X170G - 582825
6,7,30-IOGURTE INT MAMAO VIGOR 12X170G - 582825
7,8,35-IOGURTE INT MAMAO VIGOR 12X170G - 582825
8,9,10-IOGURTE INT MAMAO VIGOR 12X170G - 582825
9,10,13-IOGURTE INT MAMAO VIGOR 12X170G - 582825



C. Prova de Robustez (Cosméticos)
Consulta: SHAMPOO ELSEVE OLEO EXTRAORDINARIO NUTRICAO 400ML


,rank,XPROD_recuperado
0,1,COND ELSEVE NUTRICAO 10X 400ML
1,2,SH ELSEVE OLEO EXTRAORD NUTRICAO 1X400ML RH132...
2,3,OLEO EXTRAORDINARIO DISCIPLINANTE:100ml
3,4,SH ELSEVE 400ML COLORVIVE
4,5,SH ELSEVE SUPREME CONTROL 4D 400ML
5,6,SH ELSEVE 400ML ARGININA RESIST X3
6,7,REVITRAT NUTRICAO SHAMPOO - 230ML -L.201407068
7,8,OLEO CAP ELSEVE EXTRAORD RESS100ML
8,9,BC OLEO ARGAN MASC NUTRICAO 60G
9,10,SHAMPOO ELSEVE LISS INTENSE EXTEME 200ML


## 5. Bloco 2 e 3: LLM, Contrato de Dados e Determinismo

Com o lote semântico pronto, iniciamos a extração estruturada com foco em **confiabilidade de produção**.

Nesta etapa adotamos três princípios:

1. **Segurança operacional:** credenciais por variável de ambiente (`.env`), sem chave hardcoded.
2. **Contrato estrito de saída:** uso de `response_format={"type": "json_object"}` para garantir JSON válido no formato esperado.
3. **Determinismo:** `temperature = 0` para reduzir variância e facilitar auditoria, comparação de versões e benchmark.

O objetivo é simples: transformar texto fiscal ruidoso em dados estruturados, mantendo consistência e previsibilidade entre execuções.

In [22]:
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()


class ProductionFiscalLLMEngine:
    """Motor de inferência conectado à API da OpenAI (Bloco 3)."""

    def __init__(self, model_name: str = "gpt-4o"):
        self.client = OpenAI()
        self.model_name = model_name

    def _build_system_prompt(self) -> str:
        """Prompt de sistema aderente ao contrato de normalização fiscal."""
        return """
        [SYSTEM]
        Você é um algoritmo especialista governamental em extração e normalização de atributos fiscais a partir de descrições curtas e ruidosas de mercadoria.
        Sua única função é processar o lote de descrições fornecidas e retornar um objeto JSON estruturado. Não adicione comentários.

        [TASK & RULES]
        Para cada descrição do lote, execute as 3 operações:
        1. Name Expansion: Expanda siglas.
        2. String Wrangling: Remova códigos de lote, validades, caracteres de rastreabilidade e corrija ortografia (Title Case).
        3. UoM Normalization: Identifique a quantidade e a unidade de medida do produto unitário. Converta para o padrão SEFAZ (ML -> Mililitros, G -> Gramas, UN -> Unidade, KG -> Quilogramas, L -> Litros).

        RESTRIÇÕES CRÍTICAS ABSOLUTAS:
        - NUNCA multiplique valores de embalagens ou fardos.
        - Se o texto diz "10X400ML", a quantidade do produto é 400. NUNCA 4000.
        - Se o texto diz "CX 12X1L", a quantidade é 1. NUNCA 12.
        - Mantenha o Nome Normalizado o mais fiel possível ao original, sem inventar palavras.

        [OUTPUT SCHEMA]
        Retorne estritamente um JSON com a seguinte estrutura raiz contendo a lista dos itens processados:
        {
          "produtos_normalizados": [
             {
               "Nome_Normalizado": "string",
               "Quantidade": number,
               "Unidade_Medida": "string"
             }
          ]
        }

        [FEW-SHOT EXAMPLES]
        Entrada: "REFRIG COCA COLA LT 350ML"
        Saída: {"Nome_Normalizado": "Refrigerante Coca-Cola", "Quantidade": 350, "Unidade_Medida": "Mililitros"}

        Entrada: "DIPIRONA SOD 500MG/ML GTS 10ML"
        Saída: {"Nome_Normalizado": "Dipirona Sódica", "Quantidade": 10, "Unidade_Medida": "Mililitros"}

        Entrada: "QUEIJO MINAS FRESCAL KG"
        Saída: {"Nome_Normalizado": "Queijo Minas Frescal", "Quantidade": 1, "Unidade_Medida": "Quilogramas"}

        Entrada: "SABONETE LIQ DOVE HIDRAT FRS 250ML"
        Saída: {"Nome_Normalizado": "Sabonete Líquido Dove Hidratante", "Quantidade": 250, "Unidade_Medida": "Mililitros"}

        Entrada: "LEITE INTEGRAL LONGA VIDA 1LT CX 12UN"
        Saída: {"Nome_Normalizado": "Leite Integral Longa Vida", "Quantidade": 12, "Unidade_Medida": "Unidade"}
        """

    @staticmethod
    def _build_user_prompt(batch: list[str]) -> str:
        """Formata o lote BM25 no formato esperado pela LLM."""
        prompt = "[USER]\nAnalise o seguinte lote de descrições fiscais e retorne o JSON correspondente:\n\n"
        for i, item in enumerate(batch, 1):
            prompt += f"Item {i}: {item}\n"
        return prompt

    def process_batch(self, batch: list[str]) -> dict:
        """Executa a requisição para a LLM e retorna o JSON parseado."""
        if not batch:
            return {"produtos_normalizados": []}

        print(f"Enviando lote de {len(batch)} itens para o modelo {self.model_name}...")

        response = self.client.chat.completions.create(
            model=self.model_name,
            messages=[
                {"role": "system", "content": self._build_system_prompt()},
                {"role": "user", "content": self._build_user_prompt(batch)}
            ],
            temperature=0,
            response_format={"type": "json_object"}
        )

        raw_content = response.choices[0].message.content
        return json.loads(raw_content)

llm_engine = ProductionFiscalLLMEngine(model_name="gpt-4o")

lote_shampoo = blocker.get_batch_for(
    query_text="SHAMPOO ELSEVE OLEO EXTRAORDINARIO NUTRICAO 400ML",
    batch_size=15
)

resultado_json = llm_engine.process_batch(lote_shampoo)

print("\nResultado da normalização semântica\n")
produtos = resultado_json.get("produtos_normalizados", [])

for i, original in enumerate(lote_shampoo):
    print(f"ORIGINAL (XPROD): {original}")
    if i < len(produtos):
        prod = produtos[i]
        print(
            f"NORMALIZADO: Nome: '{prod.get('Nome_Normalizado')}' "
            f"| Qtd: {prod.get('Quantidade')} "
            f"| UoM: '{prod.get('Unidade_Medida')}'"
        )
    print("-" * 80)

Enviando lote de 15 itens para o modelo gpt-4o...

Resultado da normalização semântica

ORIGINAL (XPROD): COND ELSEVE NUTRICAO 10X 400ML
NORMALIZADO: Nome: 'Condicionador Elseve Nutrição' | Qtd: 400 | UoM: 'Mililitros'
--------------------------------------------------------------------------------
ORIGINAL (XPROD): SH ELSEVE OLEO EXTRAORD NUTRICAO 1X400ML RH1325600
NORMALIZADO: Nome: 'Shampoo Elseve Óleo Extraordinário Nutrição' | Qtd: 400 | UoM: 'Mililitros'
--------------------------------------------------------------------------------
ORIGINAL (XPROD): OLEO EXTRAORDINARIO DISCIPLINANTE:100ml
NORMALIZADO: Nome: 'Óleo Extraordinário Disciplinante' | Qtd: 100 | UoM: 'Mililitros'
--------------------------------------------------------------------------------
ORIGINAL (XPROD): SH ELSEVE 400ML COLORVIVE
NORMALIZADO: Nome: 'Shampoo Elseve Colorvive' | Qtd: 400 | UoM: 'Mililitros'
--------------------------------------------------------------------------------
ORIGINAL (XPROD): SH ELSEVE

## 6. Bloco 4: Resiliência e Governança de Qualidade

Em cenários reais, integrar LLM sem validação é arriscado. Por isso, o pipeline adiciona uma camada defensiva com:

- verificação de estrutura JSON e completude do lote;
- detecção de sinais de truncamento (*lazy output*);
- retentativas com *backoff* para falhas transitórias.

Essa camada separa uma demonstração acadêmica de uma PoC pronta para produção assistida.

In [23]:
import time


class OutputValidator:
    """Validador determinístico do Bloco 4 (anti-lazy e anti-schema-break)."""

    @staticmethod
    def validate(lote_input: list[str], json_output: dict) -> tuple[bool, str]:
        if "produtos_normalizados" not in json_output:
            return False, "Falta a chave raiz 'produtos_normalizados'."

        produtos = json_output["produtos_normalizados"]
        if not isinstance(produtos, list):
            return False, "'produtos_normalizados' deve ser uma lista."

        if len(produtos) != len(lote_input):
            return False, (
                f"Completude falhou: enviado {len(lote_input)} item(ns), "
                f"recebido {len(produtos)}."
            )

        chaves_obrigatorias = {"Nome_Normalizado", "Quantidade", "Unidade_Medida"}

        for i, prod in enumerate(produtos, start=1):
            if not isinstance(prod, dict):
                return False, f"Schema falhou no item {i}: item não é objeto JSON."

            if not chaves_obrigatorias.issubset(prod.keys()):
                return False, f"Schema falhou no item {i}: chaves obrigatórias ausentes."

            nome = str(prod.get("Nome_Normalizado", ""))
            if "..." in nome:
                return False, f"Lazy evaluation detectada no item {i}: reticências no nome."

        return True, "Validação aprovada."


class ResilientFiscalLLMEngine(ProductionFiscalLLMEngine):
    """Extensão resiliente do motor LLM com validação e retentativas."""

    def process_batch_with_retry(
        self,
        batch: list[str],
        max_retries: int = 3,
        backoff_seconds: float = 2.0
    ) -> dict:
        tentativa = 1

        while tentativa <= max_retries:
            print(f"Tentativa {tentativa}/{max_retries} de processamento do lote...")

            try:
                resultado_json = self.process_batch(batch)
                is_valid, mensagem = OutputValidator.validate(batch, resultado_json)

                if is_valid:
                    print("Bloco 4 aprovou o JSON.")
                    return resultado_json

                print(f"Bloco 4 rejeitou o lote. Motivo: {mensagem}")

            except Exception as e:
                print(f"Erro de conexão/API: {str(e)}")

            if tentativa < max_retries:
                espera = backoff_seconds * tentativa
                print(f"Backoff de {espera:.1f}s antes da próxima tentativa...")
                time.sleep(espera)

            tentativa += 1

        print("Falha crítica: lote estourou o limite de retentativas.")
        return {"produtos_normalizados": [], "status": "failed"}


# ==========================================
# TESTE: PIPELINE BLINDADO
# ==========================================

motor_resiliente = ResilientFiscalLLMEngine(model_name="gpt-4o")
resultado_seguro = motor_resiliente.process_batch_with_retry(lote_shampoo)

print("\nExtração final concluída")
for prod in resultado_seguro.get("produtos_normalizados", []):
    print(prod)

Tentativa 1/3 de processamento do lote...
Enviando lote de 15 itens para o modelo gpt-4o...
Bloco 4 aprovou o JSON.

Extração final concluída
{'Nome_Normalizado': 'Condicionador Elseve Nutrição', 'Quantidade': 400, 'Unidade_Medida': 'Mililitros'}
{'Nome_Normalizado': 'Shampoo Elseve Óleo Extraordinário Nutrição', 'Quantidade': 400, 'Unidade_Medida': 'Mililitros'}
{'Nome_Normalizado': 'Óleo Extraordinário Disciplinante', 'Quantidade': 100, 'Unidade_Medida': 'Mililitros'}
{'Nome_Normalizado': 'Shampoo Elseve Colorvive', 'Quantidade': 400, 'Unidade_Medida': 'Mililitros'}
{'Nome_Normalizado': 'Shampoo Elseve Supreme Control 4D', 'Quantidade': 400, 'Unidade_Medida': 'Mililitros'}
{'Nome_Normalizado': 'Shampoo Elseve Arginina Resist X3', 'Quantidade': 400, 'Unidade_Medida': 'Mililitros'}
{'Nome_Normalizado': 'Revitrat Nutrição Shampoo', 'Quantidade': 230, 'Unidade_Medida': 'Mililitros'}
{'Nome_Normalizado': 'Óleo Capilar Elseve Extraordinário', 'Quantidade': 100, 'Unidade_Medida': 'Mililitro

## 6.1 Módulo de Avaliação Científica

A etapa seguinte formaliza a mensuração de qualidade por atributo, com *fuzzy matching* para nomes e checagem estrita para quantidade.

A proposta aqui é separar claramente:

- **erro semântico** (nome incorreto),
- **erro numérico** (quantidade divergente),
- **erro de unidade** (UoM inconsistente).

Isso permite atacar o problema com ações direcionadas no prompt e nas regras de pós-processamento.

In [24]:
import pandas as pd
import numpy as np
import difflib

class ScientificBenchmark:
    """Avaliador estatístico com Nome por fuzzy e UoM estrita/canônica."""

    def __init__(self, ground_truth: list, llm_predictions: list, xprod_originals: list, threshold: float = 0.85):
        self.truth = ground_truth
        self.preds = llm_predictions
        self.originals = xprod_originals
        self.atributos = ["Nome_Normalizado", "Quantidade", "Unidade_Medida"]
        self.threshold = threshold  # usado apenas para Nome_Normalizado
        self.metrics = {attr: {"TP": 0, "FP": 0, "FN": 0} for attr in self.atributos}
        self.error_log = []

    def _normalize_string(self, value) -> str:
        if pd.isna(value) or value is None or str(value).strip() == "":
            return "N/A"
        return str(value).strip().lower()

    def _normalize_uom_strict(self, value) -> str:
        raw = self._normalize_string(value)
        if raw == "N/A":
            return raw

        raw = raw.replace(".", "").strip()
        aliases = {
            "ml": "mililitros", "mililitro": "mililitros", "mililitros": "mililitros",
            "l": "litros", "lt": "litros", "litro": "litros", "litros": "litros",
            "g": "gramas", "gr": "gramas", "grama": "gramas", "gramas": "gramas",
            "kg": "quilogramas", "quilo": "quilogramas", "quilograma": "quilogramas", "quilogramas": "quilogramas",
            "un": "unidade", "und": "unidade", "unid": "unidade", "unidade": "unidade", "unidades": "unidade",
        }
        return aliases.get(raw, raw)

    def _is_semantically_similar(self, str1: str, str2: str) -> bool:
        if str1 == "N/A" or str2 == "N/A":
            return False
        similaridade = difflib.SequenceMatcher(None, str1, str2).ratio()
        return similaridade >= self.threshold

    def run_evaluation(self):
        total_avaliado = min(len(self.truth), len(self.preds))

        for i in range(total_avaliado):
            esperado = self.truth[i]
            obtido = self.preds[i]
            xprod = self.originals[i] if i < len(self.originals) else "Desconhecido"

            val_esp_nome = self._normalize_string(esperado.get("Nome_Normalizado"))
            val_obt_nome = self._normalize_string(obtido.get("Nome_Normalizado"))

            if self._is_semantically_similar(val_esp_nome, val_obt_nome):
                self.metrics["Nome_Normalizado"]["TP"] += 1
            else:
                self.metrics["Nome_Normalizado"]["FP" if val_obt_nome != "N/A" else "FN"] += 1
                self._log_error(i, xprod, "Nome_Normalizado", val_esp_nome, val_obt_nome, "Erro/Omissão")

            val_esp_qtd = self._normalize_string(esperado.get("Quantidade"))
            val_obt_qtd = self._normalize_string(obtido.get("Quantidade"))
            try:
                acertou_qtd = (float(val_esp_qtd) == float(val_obt_qtd))
            except ValueError:
                acertou_qtd = (val_esp_qtd == val_obt_qtd)

            if acertou_qtd:
                self.metrics["Quantidade"]["TP"] += 1
            else:
                self.metrics["Quantidade"]["FP" if val_obt_qtd != "N/A" else "FN"] += 1
                self._log_error(i, xprod, "Quantidade", val_esp_qtd, val_obt_qtd, "Divergência Numérica")

            val_esp_uom = self._normalize_uom_strict(esperado.get("Unidade_Medida"))
            val_obt_uom = self._normalize_uom_strict(obtido.get("Unidade_Medida"))
            if val_esp_uom == val_obt_uom:
                self.metrics["Unidade_Medida"]["TP"] += 1
            else:
                self.metrics["Unidade_Medida"]["FP" if val_obt_uom != "N/A" else "FN"] += 1
                self._log_error(i, xprod, "Unidade_Medida", val_esp_uom, val_obt_uom, "Erro UoM (Estrito)")

    def _log_error(self, id_linha, xprod, atributo, esperado, obtido, tipo):
        self.error_log.append({
            "ID_Linha": id_linha + 1,
            "XPROD_Original": xprod,
            "Atributo_Falho": atributo,
            "Valor_Esperado": esperado,
            "Valor_Obtido": obtido,
            "Motivo": tipo
        })

    def generate_report(self) -> pd.DataFrame:
        print("=" * 65)
        print(f"RELATÓRIO DE VALIDAÇÃO ESTATÍSTICA (Nome Fuzzy: {self.threshold:.0%} | UoM: Estrita)")
        print("=" * 65)
        print(f"{'Atributo':<20} | {'Precisão':<9} | {'Recall':<9} | {'F1-Score':<9}")
        print("-" * 65)

        f1_scores = []
        for attr in self.atributos:
            tp = self.metrics[attr]["TP"]
            fp = self.metrics[attr]["FP"]
            fn = self.metrics[attr]["FN"]

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

            f1_scores.append(f1)
            print(f"{attr:<20} | {precision:>8.2%} | {recall:>8.2%} | {f1:>8.2%}")

        print("-" * 65)
        print(f"MACRO F1-SCORE DA ARQUITETURA: {np.mean(f1_scores):.2%}")
        print("=" * 65)
        return pd.DataFrame(self.error_log)

In [25]:
# ==========================================
# 7.0 GERAÇÃO DO TEMPLATE DE GABARITO (1x)
# ==========================================

from pathlib import Path

DATA_PATH = Path('../data/AmostraDescMerc.csv')
ARQUIVO_GABARITO = Path('gabarito_vazio_161.csv')
N_ITENS_BENCHMARK = 161
SEED = 42

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Base não encontrada: {DATA_PATH.resolve()}")

df_sefaz = pd.read_csv(DATA_PATH, sep='\t')

print(f"Sorteando {N_ITENS_BENCHMARK} itens para benchmark reprodutível...")

df_amostra_161 = df_sefaz.sample(n=N_ITENS_BENCHMARK, random_state=SEED).copy()
df_amostra_161['Nome_Normalizado'] = ''
df_amostra_161['Quantidade'] = ''
df_amostra_161['Unidade_Medida'] = ''

df_amostra_161 = df_amostra_161[['Text', 'Nome_Normalizado', 'Quantidade', 'Unidade_Medida']]
df_amostra_161.rename(columns={'Text': 'XPROD_Original'}, inplace=True)

df_amostra_161.to_csv(ARQUIVO_GABARITO, index=False, encoding='utf-8-sig', sep=';')

print(f"Template criado: {ARQUIVO_GABARITO.resolve()}")
print("Próximo passo: preencher manualmente as 3 colunas e salvar o arquivo.")

Sorteando 161 itens para benchmark reprodutível...
Template criado: /Users/brennohdev/development/NLP-tail/SEFAZ/PoC-AD/notebooks/gabarito_vazio_161.csv
Próximo passo: preencher manualmente as 3 colunas e salvar o arquivo.


## 7. Benchmark Científico Final (161 itens) — Protocolo de Produção

Nesta fase final, avaliamos a arquitetura em uma amostra cega de 161 registros com foco em **evidência estatística**.

### Protocolo

1. gerar template de gabarito com amostragem reprodutível (seed fixa);
2. preencher rotulagem humana (`Nome_Normalizado`, `Quantidade`, `Unidade_Medida`);
3. rodar inferência em lotes de 15 com retentativas e validação;
4. medir desempenho por atributo e inspecionar log de erro.

### Critério de qualidade

A PoC é considerada robusta quando o relatório final permite:

- identificar causa dominante de falha (nome, quantidade ou unidade);
- comparar versões de prompt de forma justa;
- orientar decisões de melhoria com base em dados observáveis.

In [26]:
import pandas as pd
import time
from pathlib import Path

# ==========================================
# 1. CARREGAMENTO ROBUSTO DO GABARITO HUMANO
# ==========================================

def carregar_gabarito(path_csv: str):
    """Carrega, valida e normaliza o CSV de gabarito."""
    path = Path(path_csv)
    if not path.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {path.resolve()} | "
            "Confirme se o CSV está na pasta do notebook."
        )

    df = pd.read_csv(path, sep=';')

    colunas_obrigatorias = {
        'XPROD_Original',
        'Nome_Normalizado',
        'Quantidade',
        'Unidade_Medida'
    }
    faltantes = colunas_obrigatorias.difference(df.columns)
    if faltantes:
        raise ValueError(f"Colunas obrigatórias ausentes no gabarito: {sorted(faltantes)}")

    # Padroniza ausências para não quebrar benchmark
    df = df.copy()
    df['XPROD_Original'] = df['XPROD_Original'].fillna('').astype(str)
    df['Nome_Normalizado'] = df['Nome_Normalizado'].fillna('N/A')
    df['Quantidade'] = df['Quantidade'].fillna('N/A')
    df['Unidade_Medida'] = df['Unidade_Medida'].fillna('N/A')

    xprods = df['XPROD_Original'].tolist()
    gabarito = [
        {
            'Nome_Normalizado': row['Nome_Normalizado'],
            'Quantidade': row['Quantidade'],
            'Unidade_Medida': row['Unidade_Medida']
        }
        for _, row in df.iterrows()
    ]

    return xprods, gabarito, df


def inferir_em_lotes(engine, xprods: list, tamanho_lote: int = 15, pausa_segundos: float = 2.0):
    """Executa inferência em lotes com proteção de completude."""
    todas_predicoes = []

    for i in range(0, len(xprods), tamanho_lote):
        lote_atual = xprods[i : i + tamanho_lote]
        num_lote = (i // tamanho_lote) + 1
        print(f"Processando Lote {num_lote} (Itens {i + 1} a {i + len(lote_atual)})...")

        resultado = engine.process_batch_with_retry(lote_atual)
        produtos_extraidos = resultado.get('produtos_normalizados', [])

        # Boas práticas: garantir 1:1 entre entrada e saída para avaliação justa
        if len(produtos_extraidos) < len(lote_atual):
            deficit = len(lote_atual) - len(produtos_extraidos)
            print(f"Lote {num_lote}: retorno incompleto ({len(produtos_extraidos)}/{len(lote_atual)}). Padding com N/A.")
            produtos_extraidos.extend([
                {'Nome_Normalizado': 'N/A', 'Quantidade': 'N/A', 'Unidade_Medida': 'N/A'}
                for _ in range(deficit)
            ])

        todas_predicoes.extend(produtos_extraidos[:len(lote_atual)])

        # pausa apenas entre lotes
        if i + tamanho_lote < len(xprods):
            time.sleep(pausa_segundos)

    return todas_predicoes


print("Carregando o gabarito (ground truth)...")

try:
    xprods_161_itens, gabarito_161_itens, df_gabarito = carregar_gabarito('gabarito_vazio_161 copy.csv')
except (FileNotFoundError, ValueError) as e:
    print(f"Erro: {e}")
    raise

print(f"{len(xprods_161_itens)} itens carregados com sucesso.\n")

# ==========================================
# 2. ORQUESTRAÇÃO DA INFERÊNCIA NO LLM
# ==========================================
print("Iniciando inferência no LLM (pode levar alguns minutos)...")

motor_resiliente = ResilientFiscalLLMEngine()
todas_predicoes = inferir_em_lotes(
    engine=motor_resiliente,
    xprods=xprods_161_itens,
    tamanho_lote=15,
    pausa_segundos=2.0
)

print("\nExtração concluída. Cruzando com o gabarito...\n")

# ==========================================
# 3. AVALIAÇÃO ESTATÍSTICA + ANÁLISE DE ERRO
# ==========================================
benchmarker_fuzzy = ScientificBenchmark(
    ground_truth=gabarito_161_itens,
    llm_predictions=todas_predicoes,
    xprod_originals=xprods_161_itens,
    threshold=0.85
)

benchmarker_fuzzy.run_evaluation()
df_erros = benchmarker_fuzzy.generate_report()

print("\nLog de erros (onde o modelo falhou e penalizou o F1):")
display(df_erros)

# Resumo executivo de falhas
if not df_erros.empty:
    print("\nTop 10 atributos com mais falhas:")
    display(
        df_erros['Atributo_Falho']
        .value_counts()
        .rename_axis('Atributo')
        .reset_index(name='Qtde_Erros')
        .head(10)
    )
else:
    print("\nNenhuma falha registrada no log de erros.")

Carregando o gabarito (ground truth)...
161 itens carregados com sucesso.

Iniciando inferência no LLM (pode levar alguns minutos)...
Processando Lote 1 (Itens 1 a 15)...
Tentativa 1/3 de processamento do lote...
Enviando lote de 15 itens para o modelo gpt-4o...
Bloco 4 aprovou o JSON.
Processando Lote 2 (Itens 16 a 30)...
Tentativa 1/3 de processamento do lote...
Enviando lote de 15 itens para o modelo gpt-4o...
Bloco 4 aprovou o JSON.
Processando Lote 3 (Itens 31 a 45)...
Tentativa 1/3 de processamento do lote...
Enviando lote de 15 itens para o modelo gpt-4o...
Bloco 4 aprovou o JSON.
Processando Lote 4 (Itens 46 a 60)...
Tentativa 1/3 de processamento do lote...
Enviando lote de 15 itens para o modelo gpt-4o...
Bloco 4 aprovou o JSON.
Processando Lote 5 (Itens 61 a 75)...
Tentativa 1/3 de processamento do lote...
Enviando lote de 15 itens para o modelo gpt-4o...
Bloco 4 aprovou o JSON.
Processando Lote 6 (Itens 76 a 90)...
Tentativa 1/3 de processamento do lote...
Enviando lote de 

,ID_Linha,XPROD_Original,Atributo_Falho,Valor_Esperado,Valor_Obtido,Motivo
0,8,TRESEM CD HID PROF 200,Nome_Normalizado,condicionador tresemmé hidratação profunda,tresemmé condicionador hidratante profissional,Erro/Omissão
1,10,PRO - LAPIS CARBON N. 10 - MARINHO,Nome_Normalizado,lápis carbono marinho,pro lápis carbono número 10 marinho,Erro/Omissão
2,19,"CHA LEAO FUZE ROS+HIB+AMO 1,6GX15UN CX36",Quantidade,1.6,15,Divergência Numérica
3,19,"CHA LEAO FUZE ROS+HIB+AMO 1,6GX15UN CX36",Unidade_Medida,gramas,unidade,Erro UoM (Estrito)
4,20,ACIDO SULFONICO 90% (TEBRAS),Nome_Normalizado,ácido sulfônico 90% tebras,ácido sulfônico 90%,Erro/Omissão
...,...,...,...,...,...,...
61,157,4311906 ST JG COZINHA 3 PC LT VD,Nome_Normalizado,jogo de cozinha,jogo cozinha 3 peças,Erro/Omissão
62,157,4311906 ST JG COZINHA 3 PC LT VD,Quantidade,3.0,1,Divergência Numérica
63,159,(4673) Bettabanho Lev 3 Pag 2 1x24CJ,Nome_Normalizado,bettabanho leve 3 pague 2,bettabanho,Erro/Omissão
64,159,(4673) Bettabanho Lev 3 Pag 2 1x24CJ,Quantidade,24.0,1,Divergência Numérica



Top 10 atributos com mais falhas:


,Atributo,Qtde_Erros
0,Nome_Normalizado,39
1,Quantidade,14
2,Unidade_Medida,13


## 7.1 Benchmark de Tokens, Custo e Eficiência

Para medir se a arquitetura está otimizada, monitore quatro dimensões por lote:

1. **Tokens**: `prompt_tokens`, `completion_tokens`, `total_tokens`.
2. **Custo estimado (USD)** usando preço por **1 milhão** de tokens:
   $$
   \text{Custo} = \frac{T_{in}}{10^6}\cdot P_{in} + \frac{T_{out}}{10^6}\cdot P_{out}
   $$
3. **Latência**: média, P50 e P95 por lote.
4. **Eficiência**: tokens por item e custo por item processado.

Premissas de preço configuradas neste notebook (conforme referência informada):

- GPT-4 Turbo: input = US$ 10.00 / 1M, output = US$ 30.00 / 1M
- GPT-4 (8k): input = US$ 30.00 / 1M, output = US$ 60.00 / 1M
- GPT-4 (32k): input = US$ 60.00 / 1M, output = US$ 120.00 / 1M

Além do LLM, também medimos o custo computacional do **Bloco 1 (BM25)** em milissegundos por consulta para separar gargalo de recuperação vs gargalo de inferência.

In [27]:
import time
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# PREÇOS DE REFERÊNCIA (USD por 1 milhão de tokens)
# ---------------------------------------------------------
MODEL_PRICING_USD_PER_1M = {
    'gpt-4-turbo': {'input': 10.0, 'output': 30.0},
    'gpt-4-8k': {'input': 30.0, 'output': 60.0},
    'gpt-4': {'input': 30.0, 'output': 60.0},
    'gpt-4-32k': {'input': 60.0, 'output': 120.0},
}

DEFAULT_PRICING_USD_PER_1M = {'input': 0.0, 'output': 0.0}


def get_pricing_for_model(model_name: str) -> dict:
    """Retorna pricing por 1M tokens para o modelo informado."""
    if model_name in MODEL_PRICING_USD_PER_1M:
        return MODEL_PRICING_USD_PER_1M[model_name]

    # fallback por prefixo (ex.: gpt-4-turbo-2024-xx)
    for key, val in MODEL_PRICING_USD_PER_1M.items():
        if model_name.startswith(key):
            return val

    return DEFAULT_PRICING_USD_PER_1M


def _usage_value(usage_obj, field_name: str) -> int:
    """Compatibilidade para objetos usage em formatos diferentes."""
    if usage_obj is None:
        return 0

    if hasattr(usage_obj, field_name):
        value = getattr(usage_obj, field_name)
        return int(value or 0)

    if isinstance(usage_obj, dict):
        return int(usage_obj.get(field_name, 0) or 0)

    return 0


def estimate_cost_usd(prompt_tokens: int, completion_tokens: int, pricing_per_1m: dict) -> float:
    """Calcula custo com base em preço por 1M tokens."""
    return (
        (prompt_tokens / 1_000_000.0) * pricing_per_1m.get('input', 0.0)
        + (completion_tokens / 1_000_000.0) * pricing_per_1m.get('output', 0.0)
    )


def benchmark_bm25_latency(blocker_obj, queries: list[str], batch_size: int = 15) -> pd.DataFrame:
    """Mede latência do Bloco 1 (recuperação lexical)."""
    rows = []

    for i, q in enumerate(queries, start=1):
        t0 = time.perf_counter()
        _ = blocker_obj.get_batch_for(query_text=q, batch_size=batch_size)
        latency_ms = (time.perf_counter() - t0) * 1000.0

        rows.append({
            'query_id': i,
            'query_chars': len(q),
            'bm25_latency_ms': latency_ms
        })

    return pd.DataFrame(rows)


def benchmark_llm_tokens_and_cost(
    engine,
    textos: list[str],
    batch_size: int = 15,
    max_batches: int = 8,
    sleep_seconds: float = 0.5,
    pricing_per_1m: dict | None = None
) -> pd.DataFrame:
    """Mede tokens, latência e custo estimado por lote no LLM."""
    pricing_per_1m = pricing_per_1m or get_pricing_for_model(engine.model_name)
    rows = []

    total_disponivel = len(textos)
    if total_disponivel == 0:
        return pd.DataFrame()

    total_lotes = min(max_batches, int(np.ceil(total_disponivel / batch_size)))

    for b in range(total_lotes):
        ini = b * batch_size
        fim = min((b + 1) * batch_size, total_disponivel)
        lote = textos[ini:fim]

        if not lote:
            continue

        t0 = time.perf_counter()
        response = engine.client.chat.completions.create(
            model=engine.model_name,
            messages=[
                {'role': 'system', 'content': engine._build_system_prompt()},
                {'role': 'user', 'content': engine._build_user_prompt(lote)}
            ],
            temperature=0,
            response_format={'type': 'json_object'}
        )
        latency_s = time.perf_counter() - t0

        usage = getattr(response, 'usage', None)
        prompt_tokens = _usage_value(usage, 'prompt_tokens')
        completion_tokens = _usage_value(usage, 'completion_tokens')
        total_tokens = _usage_value(usage, 'total_tokens')

        if total_tokens == 0:
            total_tokens = prompt_tokens + completion_tokens

        cost_usd = estimate_cost_usd(prompt_tokens, completion_tokens, pricing_per_1m)

        rows.append({
            'batch_id': b + 1,
            'model_name': engine.model_name,
            'batch_size_real': len(lote),
            'prompt_tokens': prompt_tokens,
            'completion_tokens': completion_tokens,
            'total_tokens': total_tokens,
            'latency_s': latency_s,
            'tokens_per_item': total_tokens / max(len(lote), 1),
            'cost_usd': cost_usd,
            'cost_usd_per_item': cost_usd / max(len(lote), 1)
        })

        if b < total_lotes - 1:
            time.sleep(sleep_seconds)

    return pd.DataFrame(rows)


def summarize_benchmark(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame()

    def p95(series):
        return float(np.percentile(series, 95))

    summary = {
        f'{prefix}_n_lotes': len(df),
        f'{prefix}_latency_mean_s': df['latency_s'].mean() if 'latency_s' in df else np.nan,
        f'{prefix}_latency_p50_s': df['latency_s'].median() if 'latency_s' in df else np.nan,
        f'{prefix}_latency_p95_s': p95(df['latency_s']) if 'latency_s' in df else np.nan,
        f'{prefix}_tokens_total': df['total_tokens'].sum() if 'total_tokens' in df else np.nan,
        f'{prefix}_tokens_per_item_mean': df['tokens_per_item'].mean() if 'tokens_per_item' in df else np.nan,
        f'{prefix}_cost_usd_total': df['cost_usd'].sum() if 'cost_usd' in df else np.nan,
        f'{prefix}_cost_usd_per_item_mean': df['cost_usd_per_item'].mean() if 'cost_usd_per_item' in df else np.nan,
    }

    return pd.DataFrame([summary])


# ===============================
# EXECUÇÃO DO BENCHMARK
# ===============================

# 1) BM25: custo computacional local (CPU)
queries_bm25 = df_sefaz['Text'].dropna().astype(str).sample(n=50, random_state=42).tolist()
df_bm25_bench = benchmark_bm25_latency(blocker, queries_bm25, batch_size=15)

print('Benchmark BM25 (latência em ms por consulta):')
display(df_bm25_bench.head())
print(
    f"BM25 | mean={df_bm25_bench['bm25_latency_ms'].mean():.2f}ms "
    f"| p50={df_bm25_bench['bm25_latency_ms'].median():.2f}ms "
    f"| p95={np.percentile(df_bm25_bench['bm25_latency_ms'], 95):.2f}ms"
)

# 2) LLM: tokens, latência e custo
amostra_llm = xprods_161_itens[:120]  # recorte opcional para acelerar benchmark
pricing_modelo_ativo = get_pricing_for_model(llm_engine.model_name)

print(f"\nPricing aplicado para {llm_engine.model_name}: {pricing_modelo_ativo} USD / 1M tokens")

df_llm_bench = benchmark_llm_tokens_and_cost(
    engine=llm_engine,
    textos=amostra_llm,
    batch_size=15,
    max_batches=8,
    sleep_seconds=0.7,
    pricing_per_1m=pricing_modelo_ativo
)

print('\nBenchmark LLM (tokens/custo/latência por lote):')
display(df_llm_bench)

resumo_llm = summarize_benchmark(df_llm_bench, prefix='llm')
print('\nResumo executivo do benchmark LLM:')
display(resumo_llm)

if pricing_modelo_ativo['input'] == 0.0 and pricing_modelo_ativo['output'] == 0.0:
    print('Modelo sem pricing configurado. Defina MODEL_PRICING_USD_PER_1M para calcular custo financeiro.')

# 3) Diagnóstico rápido de otimização
if not df_llm_bench.empty:
    lat_p95 = np.percentile(df_llm_bench['latency_s'], 95)
    tok_item = df_llm_bench['tokens_per_item'].mean()

    print('\nDiagnóstico rápido:')
    print(f'- Tokens médios por item: {tok_item:.2f}')
    print(f'- Latência P95 por lote: {lat_p95:.2f}s')
    print('- Use esses dois indicadores para comparar versões de prompt/arquitetura (A/B).')

Benchmark BM25 (latência em ms por consulta):


,query_id,query_chars,bm25_latency_ms
0,1,27,7.929791
1,2,20,4.031000
2,3,28,6.185750
3,4,49,11.040667
4,5,29,4.284250


BM25 | mean=5.69ms | p50=4.98ms | p95=10.25ms

Pricing aplicado para gpt-4o: {'input': 30.0, 'output': 60.0} USD / 1M tokens

Benchmark LLM (tokens/custo/latência por lote):


,batch_id,model_name,batch_size_real,prompt_tokens,completion_tokens,total_tokens,latency_s,tokens_per_item,cost_usd,cost_usd_per_item
0,1,gpt-4o,15,933,611,1544,8.778202,102.933333,0.06465,0.004310
1,2,gpt-4o,15,1002,609,1611,7.492277,107.400000,0.06660,0.004440
2,3,gpt-4o,15,998,615,1613,9.228675,107.533333,0.06684,0.004456
3,4,gpt-4o,15,962,600,1562,8.623096,104.133333,0.06486,0.004324
4,5,gpt-4o,15,987,614,1601,9.077482,106.733333,0.06645,0.004430
5,6,gpt-4o,15,987,606,1593,7.286970,106.200000,0.06597,0.004398
6,7,gpt-4o,15,980,606,1586,6.527703,105.733333,0.06576,0.004384
7,8,gpt-4o,15,986,612,1598,9.652869,106.533333,0.06630,0.004420



Resumo executivo do benchmark LLM:


,llm_n_lotes,llm_latency_mean_s,llm_latency_p50_s,llm_latency_p95_s,llm_tokens_total,llm_tokens_per_item_mean,llm_cost_usd_total,llm_cost_usd_per_item_mean
0,8,8.333409,8.700649,9.504401,12708,105.9,0.52743,0.004395



Diagnóstico rápido:
- Tokens médios por item: 105.90
- Latência P95 por lote: 9.50s
- Use esses dois indicadores para comparar versões de prompt/arquitetura (A/B).


## 8. Conclusões, boas práticas e próximos passos

### Síntese desta versão

1. Encadeamento técnico completo: da blocagem à validação empírica em escala (161 itens).
2. Avaliação confiável: validação de schema e tratamento explícito de valores ausentes.
3. Robustez operacional: inferência em lotes com proteção para retornos incompletos.
4. Observabilidade: relatório de erros por atributo para orientar melhorias de prompt e regras.

### Boas práticas aplicadas

- separação clara por blocos (ingestão, inferência, validação e benchmark);
- retentativas com backoff no processamento do LLM;
- uso de métricas objetivas e log de erros auditável;
- amostragem com semente fixa para reprodutibilidade.

### Evoluções recomendadas

- extrair classes para módulo Python (`src/`) e manter o notebook como camada de experimentação;
- versionar prompts por arquivo e registrar configuração de experimento (`model`, `temperature`, `threshold`);
- adicionar testes unitários para `OutputValidator` e `ScientificBenchmark`;
- persistir resultados em CSV/Parquet com timestamp para comparação histórica.